In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install telethon

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 14.2 MB/s eta 0:00:00
  Created wheel for pyaes: filename=pyaes-1.6.1-py3-none-any.whl size=26347 sha256=5d8c9c16fa1bc89d615ac7e6e9e711be6e7b183fcf0b8db633893c809ad170a8
  Stored in directory: /root/.cache/pip/wheels/4e/52/33/010d0843550bffb6a591b11629070ae140c0ad4f53e68a3bd3
Successfully built pyaes


# **Task 1: Data Ingestion and Preprocessing**

In [ ]:
from telethon.sync import TelegramClient
from telethon.tl.types import MessageMediaPhoto, MessageMediaDocument
import re
from transformers import AutoTokenizer

# Configuration
API_ID = '27006778'
API_HASH = 'f41204fef3102a1ca48d248f3c287425'
PHONE = '+251924246518'
SESSION_NAME = 'amharic_ner_session'
# Selected channels
CHANNELS = [
    'ZemenExpress',
    'Leyueqa',
    'MerttEka',
    'classybrands',
    'belaclassic',
    'AwasMart'
]

In [ ]:
async def scrape_telegram_channels():
    """Main scraping function adapted for Jupyter"""
    client = TelegramClient(SESSION_NAME, API_ID, API_HASH)
    await client.start()

    all_messages = []

    for channel in CHANNELS:
        print(f"Scraping {channel}...")
        try:
            async for message in client.iter_messages(channel, limit=10000):
                if message.text:
                    all_messages.append({
                        'channel': channel,
                        'message_id': message.id,
                        'date': message.date,
                        'views': message.views or 0,
                        'text': message.text,
                        'has_media': bool(message.media)
                    })
        except Exception as e:
            print(f"Error scraping {channel}: {e}")

    await client.disconnect()
    return pd.DataFrame(all_messages)

# In Jupyter, you can await directly:
df = await scrape_telegram_channels()
df.to_csv('telegram_data.csv', index=False)
print("Data saved successfully")
def clean_amharic_text(text):
    """Clean and normalize Amharic text"""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s\u1200-\u137F.,!?]', '', text)
    text = re.sub(r'[^\u1200-\u137F\s]', '', text)  # Keep only Amharic characters and spaces
    text = re.sub(r'\s+', ' ', text)  # Normalize spaces
    return re.sub(r'\s+', ' ', text).strip()

df['processed_text'] = df['text'].apply(
    lambda x: ' '.join(clean_amharic_text(x).split()) if pd.notnull(x) else ""
)

df.to_csv('telegram_data_processed.csv', index=False)
print("Data saved successfully")

Signed in successfully as @; remember to not break the ToS or you will risk an account ban!
Scraping ZemenExpress...
Scraping Leyueqa...
Scraping MerttEka...
Scraping classybrands...
Scraping belaclassic...
Scraping AwasMart...
Data saved successfully
Data saved successfully


In [ ]:
df=pd.read_csv('/content/telegram_data_processed.csv')

In [ ]:
#df.drop(columns=['text'],inplace=True)
df.dropna(inplace=True)
df

,channel,message_id,date,views,text,has_media,processed_text
0,ZemenExpress,7007,2025-06-24 11:49:18+00:00,2199,💥💥...................................💥💥\n\n🎯 L...,True,ለልጅዎ ልጆች እየተዝናኑ የሚማሩበት ለፅሁፍ እና ለስዕል መለማመጃ የሚሆን...
1,ZemenExpress,7006,2025-06-24 11:49:01+00:00,1781,💥💥...................................💥💥\n\n🎯 L...,True,ለልጅዎ ልጆች እየተዝናኑ የሚማሩበት ለፅሁፍ እና ለስዕል መለማመጃ የሚሆን...
2,ZemenExpress,7005,2025-06-24 11:48:41+00:00,1801,💥💥...................................💥💥\n\n🎯 L...,True,ለልጅዎ ልጆች እየተዝናኑ የሚማሩበት ለፅሁፍ እና ለስዕል መለማመጃ የሚሆን...
3,ZemenExpress,7004,2025-06-23 14:55:46+00:00,2639,💥💥👀 ...........💥💥\n\n📌 Electric Charcoal Burne...,True,በቀላሉ ከሰል ለማያያዝ የሚሆን አነስ ያለ ቦታ የማይዝ በኤሌክትሪክ የሚሰ...
4,ZemenExpress,7000,2025-06-23 14:55:40+00:00,2152,💥💥👀 ...........💥💥\n\n📌 Electric Charcoal Burne...,True,በቀላሉ ከሰል ለማያያዝ የሚሆን አነስ ያለ ቦታ የማይዝ በኤሌክትሪክ የሚሰ...
...,...,...,...,...,...,...,...
21800,AwasMart,2078,2022-08-22 12:43:55+00:00,27472,🎯 Reusable Non-Stick Silicon Baking Mat / Doug...,True,ዋጋ፦ ብር አድሚኑን ለማናገር አድራሻችን ቦሌ መድሐኔዓለም ቦሌ መሰናዶ ት...
21801,AwasMart,2077,2022-08-22 12:42:49+00:00,27376,🎯 Reusable Non-Stick Silicon Baking Mat / Doug...,True,ዋጋ፦ ብር አድሚኑን ለማናገር አድራሻችን ቦሌ መድሐኔዓለም ቦሌ መሰናዶ ት...
21804,AwasMart,2074,2022-08-22 08:33:49+00:00,15407,🎯Spa Gel Socks\n\n🔰ለእግር ልስላሴ \n🔰ለሚሰነጣጠቅ እግር\n🔰...,True,ለእግር ልስላሴ ለሚሰነጣጠቅ እግር ድርቀትን የሚያለሰልስ የሚታጠብ በቀን ...
21805,AwasMart,2073,2022-08-22 04:33:16+00:00,16304,🎯 Magic Silicone Dish Washing Gloves\n\nዋጋ፦ ...,True,ዋጋ፦ ብር አድሚኑን ለማናገር አድራሻችን ቦሌ መድሐኔዓለም ቦሌ መሰናዶ ት...


In [ ]:
df.drop(columns="text", inplace=True)

In [ ]:
df.to_csv('telegram_data_processed_task1.csv', index=False)

## **Task 2: Labeling Data in CoNLL Format**

In [ ]:
import pandas as pd
import re
from collections import defaultdict
import random

def load_data(filename='/content/telegram_data_processed_task1.csv'):
    """Load processed data with proper encoding and column validation"""
    try:
        df = pd.read_csv(filename)
        # Ensure we have the expected columns
        if 'processed_text' not in df.columns:
            if 'text' in df.columns:
                df['processed_text'] = df['text']
            else:
                raise ValueError("Input file must contain either 'processed_text' or 'text' column")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return pd.DataFrame(columns=['channel', 'message_id', 'date', 'views', 'has_media', 'processed_text'])

In [ ]:
def clean_amharic_text(text):
    """Enhanced cleaning for Amharic text"""
    if not isinstance(text, str):
        return ""
    # Remove special characters and normalize
    text = re.sub(r'[^\w\s\u1200-\u137F.,!?።፣፤፥፦፧፨]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def simple_amharic_tokenizer(text):
    """Improved tokenizer for Amharic text that handles common cases"""
    # Split on whitespace and punctuation while preserving Amharic punctuation
    tokens = re.findall(r'[\w\u1200-\u137F]+|[\.,!?።፣፤፥፦፧፨]', text)
    return tokens


In [ ]:
def label_entities(tokens, message_info=None):
    """
    Unified entity labeling interface with enhanced features
    Returns list of (token, label) tuples
    """
    labeled = []
    i = 0
    n = len(tokens)

    while i < n:
        token = tokens[i]

        # Display context and metadata
        print("\n" + "="*50)
        if message_info:
            print(f"Message ID: {message_info['message_id']} | Channel: {message_info['channel']}")
            print(f"Date: {message_info['date']} | Views: {message_info['views']}")

        print(f"\nFull text: {' '.join(tokens)}")
        print("\nCurrent context:")
        start = max(0, i-3)
        end = min(n, i+4)
        context = ' '.join(tokens[start:i] + [f'>>{tokens[i]}<<'] + tokens[i+1:end])
        print(context)

        print(f"\nCurrent token ({i+1}/{n}): {token}")
        print("\nEntity Labeling Options:")
        print("1. O (Outside)")
        print("2. B-PRODUCT (Beginning of product)")
        print("3. I-PRODUCT (Inside product)")
        print("4. B-LOC (Beginning of location)")
        print("5. I-LOC (Inside location)")
        print("6. B-PRICE (Beginning of price)")
        print("7. I-PRICE (Inside price)")
        print("8. Back to previous token")
        print("9. Skip this message")
        print("10. View current labels")
        print("11. Exit labeling")

        choice = input("Select option (1-11): ").strip()

        if choice == '1':  # O (Outside)
            labeled.append((token, 'O'))
            i += 1
        elif choice == '2':  # B-PRODUCT
            labeled.append((token, 'B-PRODUCT'))
            j = i + 1
            while j < n:
                next_token = tokens[j]
                print(f"\nNext token: {next_token}")
                cont = input("Is this part of the same product? (y/n/q to quit labeling): ").lower()
                if cont == 'q':
                    return labeled
                elif cont == 'y':
                    labeled.append((next_token, 'I-PRODUCT'))
                    j += 1
                else:
                    break
            i = j
        elif choice == '3':  # I-PRODUCT
            labeled.append((token, 'I-PRODUCT'))
            i += 1
        elif choice == '4':  # B-LOC
            labeled.append((token, 'B-LOC'))
            j = i + 1
            while j < n:
                next_token = tokens[j]
                print(f"\nNext token: {next_token}")
                cont = input("Is this part of the same location? (y/n/q to quit labeling): ").lower()
                if cont == 'q':
                    return labeled
                elif cont == 'y':
                    labeled.append((next_token, 'I-LOC'))
                    j += 1
                else:
                    break
            i = j
        elif choice == '5':  # I-LOC
            labeled.append((token, 'I-LOC'))
            i += 1
        elif choice == '6':  # B-PRICE
            labeled.append((token, 'B-PRICE'))
            j = i + 1
            while j < n:
                next_token = tokens[j]
                print(f"\nNext token: {next_token}")
                cont = input("Is this part of the same price? (y/n/q to quit labeling): ").lower()
                if cont == 'q':
                    return labeled
                elif cont == 'y':
                    labeled.append((next_token, 'I-PRICE'))
                    j += 1
                else:
                    break
            i = j
        elif choice == '7':  # I-PRICE
            labeled.append((token, 'I-PRICE'))
            i += 1
        elif choice == '8':  # Back
            if i > 0 and labeled:
                i -= 1
                labeled.pop()
        elif choice == '9':  # Skip message
            return None
        elif choice == '10':  # View labels
            print("\nCurrent labels:")
            for idx, (t, lbl) in enumerate(labeled):
                print(f"{idx+1}: {t}\t{lbl}")
            input("\nPress Enter to continue...")
        elif choice == '11':  # Exit
            return labeled
        else:
            print("Invalid choice, please try again")

    return labeled

In [ ]:
def create_conll_file(labeled_data, filename='labeled_amharic.conll'):
    """Save labeled data in CONLL format with UTF-8 encoding"""
    with open(filename, 'w', encoding='utf-8') as f:
        for sentence in labeled_data:
            if sentence:  # Skip None values (skipped messages)
                for token, label in sentence:
                    f.write(f"{token}\t{label}\n")
                f.write("\n")  # Empty line between sentences
    print(f"CONLL file saved to {filename}")

def main():
    # Load processed data
    df = load_data()

    if df.empty:
        print("Error: No data loaded. Please check your input file.")
        return

    # Clean the text data again to ensure consistency
    df['processed_text'] = df['processed_text'].apply(clean_amharic_text)

    # Select sample of messages to label
    sample_size = min(50, len(df))
    sample = df.sample(sample_size, random_state=42)

    labeled_data = []
    for idx, row in sample.iterrows():
        print(f"\n{'='*50}")
        print(f"Message {idx+1}/{sample_size}")

        # Get message metadata
        message_info = {
            'message_id': row['message_id'],
            'channel': row['channel'],
            'date': row['date'],
            'views': row['views']
        }

        # Tokenize using our improved tokenizer
        tokens = simple_amharic_tokenizer(row['processed_text'])
        print(f"\nText to label:\n{' '.join(tokens)}\n")

        labeled = label_entities(tokens, message_info)

        if labeled is not None:  # Only add if not skipped
            labeled_data.append(labeled)

        # Save progress after each message
        create_conll_file(labeled_data, 'labeled_amharic_temp.conll')

        # Show progress
        print(f"\nProgress: {len(labeled_data)}/{sample_size} messages labeled")

    # Save final CONLL file
    create_conll_file(labeled_data)
    print("\nLabeling completed successfully!")

if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
Select option (1-11): 1

Message ID: 7157 | Channel: AwasMart
Date: 2024-04-20 18:32:10+00:00 | Views: 10570

Full text: በኤሌክትሪክ የሚሰራ ሚ ሊትር ለቤት መልካም መዓዛን የሚያጎናፅፍ ባህርዛፍ ፣ጥቁርአዝሙድ ፣ሽቶ እና ሌሎችን የሚያጤሱበት ምርጥ ዕቃ ለቤትም ለቢሮም እንዲሁም ለመኝታ ቤት መጠቀም ይችላሉ። ምንም አይነት ድምፅ የሌለው የተሻለ እንቅልፍን ለማራመድ፣ ጭንቀትን ለማርገብ እና ስሜትን ለማሻሻል ጠቃሚ የሆኑ በውሃ የሚሟሟ አስፈላጊ ዘይቶችን በመጠቀም የመጨረሻውን የአሮማቴራፒ አካባቢ ይፍጠሩ። የራሱ መብራት ያለው ዋጋ፦ ብር በቴሌግራም ለማዘዝ አድራሻችን መገናኛ ኪኔሬት ሕንፃ ኛ ፎቅ ቁ ዋአች ሕንፃ ፊትለፊት ንግድ ባንክ እና ጎህ ቤቶች ባንክ ያሉበት ሜክሲኮ አልሳም አፓርታማ ግራውንድ ቁ ቦሌ መድሐኔዓለም ቦሌ መሰናዶ ትቤት ፊትለፊት አለምነሽ ፕላዛ ግራውንድ ሱቅ ቁጥር ለወዳጅዎ ስላጋሩ እናመሠግናለን ቴሌግራም ቻናል

Current context:
ጎህ ቤቶች ባንክ >>ያሉበት<< ሜክሲኮ አልሳም አፓርታማ

Current token (70/91): ያሉበት

Entity Labeling Options:
1. O (Outside)
2. B-PRODUCT (Beginning of product)
3. I-PRODUCT (Inside product)
4. B-LOC (Beginning of location)
5. I-LOC (Inside location)
6. B-PRICE (Beginning of price)
7. I-PRICE (Inside price)
8. Back to previous token
9. Skip this message
10. View current labels

KeyboardInterrupt: Interrupted by user

In [ ]:
df=pd.read_csv("/content/telegram_data_processed_task1.csv")
df

,channel,message_id,date,views,has_media,processed_text
0,ZemenExpress,7007,2025-06-24 11:49:18+00:00,2199,True,ለልጅዎ ልጆች እየተዝናኑ የሚማሩበት ለፅሁፍ እና ለስዕል መለማመጃ የሚሆን...
1,ZemenExpress,7006,2025-06-24 11:49:01+00:00,1781,True,ለልጅዎ ልጆች እየተዝናኑ የሚማሩበት ለፅሁፍ እና ለስዕል መለማመጃ የሚሆን...
2,ZemenExpress,7005,2025-06-24 11:48:41+00:00,1801,True,ለልጅዎ ልጆች እየተዝናኑ የሚማሩበት ለፅሁፍ እና ለስዕል መለማመጃ የሚሆን...
3,ZemenExpress,7004,2025-06-23 14:55:46+00:00,2639,True,በቀላሉ ከሰል ለማያያዝ የሚሆን አነስ ያለ ቦታ የማይዝ በኤሌክትሪክ የሚሰ...
4,ZemenExpress,7000,2025-06-23 14:55:40+00:00,2152,True,በቀላሉ ከሰል ለማያያዝ የሚሆን አነስ ያለ ቦታ የማይዝ በኤሌክትሪክ የሚሰ...
...,...,...,...,...,...,...
17802,AwasMart,2078,2022-08-22 12:43:55+00:00,27472,True,ዋጋ፦ ብር አድሚኑን ለማናገር አድራሻችን ቦሌ መድሐኔዓለም ቦሌ መሰናዶ ት...
17803,AwasMart,2077,2022-08-22 12:42:49+00:00,27376,True,ዋጋ፦ ብር አድሚኑን ለማናገር አድራሻችን ቦሌ መድሐኔዓለም ቦሌ መሰናዶ ት...
17804,AwasMart,2074,2022-08-22 08:33:49+00:00,15407,True,ለእግር ልስላሴ ለሚሰነጣጠቅ እግር ድርቀትን የሚያለሰልስ የሚታጠብ በቀን ...
17805,AwasMart,2073,2022-08-22 04:33:16+00:00,16304,True,ዋጋ፦ ብር አድሚኑን ለማናገር አድራሻችን ቦሌ መድሐኔዓለም ቦሌ መሰናዶ ት...


In [ ]:
def clean_amharic_text(text):
    """Clean and normalize Amharic text."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s\u1200-\u137F.,!?]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Clean the text and ensure no NaN values
df['processed_text'] = df['processed_text'].apply(clean_amharic_text)
df.dropna(subset=['processed_text'], inplace=True)
df.to_csv('telegram_data_processed_task1.csv', index=False)

# Load the processed data for labeling
def load_data(filename='telegram_data_processed_task1.csv'):
    """Load processed data with proper column validation."""
    try:
        df = pd.read_csv(filename)
        if 'processed_text' not in df.columns:
            raise ValueError("Input file must contain a 'processed_text' column")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return pd.DataFrame()

def label_entities(tokens):
    """A placeholder for entity labeling."""
    labeled = []
    for token in tokens:
        label = 'O'  # Replace with actual labeling logic
        labeled.append((token, label))
    return labeled

def create_conll_file(labeled_data, filename='labeled_amharic.conll'):
    """Save labeled data in CoNLL format."""
    with open(filename, 'w', encoding='utf-8') as f:
        for sentence in labeled_data:
            for token, label in sentence:
                f.write(f"{token}\t{label}\n")
            f.write("\n")  # Empty line between sentences

def main():
    df = load_data()
    if df.empty:
        print("No data loaded. Please check the input file.")
        return

    labeled_data = []
    for idx, row in df.iterrows():
        # Ensure that processed_text is a string
        if isinstance(row['processed_text'], str):
            tokens = row['processed_text'].split()  # Simple tokenization
            labeled = label_entities(tokens)
            labeled_data.append(labeled)
        else:
            print(f"Skipping row {idx} due to invalid processed_text.")

    create_conll_file(labeled_data)
    print("Labeling completed successfully.")

if __name__ == "__main__":
    main()

Labeling completed successfully.


In [ ]:
def merge_conll_files(file_list, output_file):
    """Merge multiple CoNLL files into a single CoNLL file."""
    with open(output_file, 'w', encoding='utf-8') as outfile:
        for filename in file_list:
            with open(filename, 'r', encoding='utf-8') as infile:
                # Read all lines from the current CoNLL file
                lines = infile.readlines()
                # Write the lines to the output file
                outfile.writelines(lines)
                # Add a newline to separate sentences from different files
                outfile.write("\n")

# Example usage
conll_files = ['/content/manually_labeled_amharic.conll', '/content/labeled_amharic_temp.conll']  # Add your CoNLL file paths
output_conll_file = 'merged_conll_file.conll'

merge_conll_files(conll_files, output_conll_file)
print(f"Merged CoNLL files into {output_conll_file}")

Merged CoNLL files into merged_conll_file.conll


In [ ]:
import pandas as pd

def read_conll_file(filename):
    """Read a CoNLL file and return a DataFrame."""
    with open(filename, 'r', encoding='utf-8') as f:
        # Prepare lists to hold tokens and labels
        tokens = []
        labels = []

        # Read the file line by line
        for line in f:
            line = line.strip()
            if line:  # If the line is not empty
                token, label = line.split('\t')  # Split by tab
                tokens.append(token)
                labels.append(label)
            else:  # Empty line indicates end of a sentence
                if tokens:  # If there are tokens collected
                    yield pd.DataFrame({'Token': tokens, 'Label': labels})
                    tokens, labels = [], []  # Reset for the next sentence

    # Handle the last sentence if it doesn't end with a new line
    if tokens:
        yield pd.DataFrame({'Token': tokens, 'Label': labels})

# Usage
conll_file_path = 'merged_conll_file.conll'  # Replace with your CoNLL file path
for sentence_df in read_conll_file(conll_file_path):
    print(sentence_df)
    print("\n---\n")  # Separator for sentences

                  Token      Label
0           በኤሌክትሪክየሚሰራ          O
1                   ለቤት          O
2                  መልካም          O
3                  መዓዛን  B-PRODUCT
4                  የሚሰጥ  I-PRODUCT
5                   ዋጋ፦      B-LOC
6                    ብር      I-LOC
7                   ውስን    B-PRICE
8                    ፍሬ    I-PRICE
9                    ነው          O
10                  ያለን  B-PRODUCT
11                 አድራሻ  I-PRODUCT
12  መገናኛመሰረትደፋርሞልሁለተኛፎቅ      B-LOC
13                   ቢሮ      I-LOC
14                    ቁ    B-PRICE
15                    በ    I-PRICE
16                 ለማዘዝ          O
17                 ይጠቀሙ  B-PRODUCT
18                ለተጨማሪ  I-PRODUCT
19                ማብራሪያ      B-LOC
20               የቴሌግራም      I-LOC
21                 ገፃችን    B-PRICE

---

    Token      Label
0    የታጠቡ          O
1     የወጥ  B-PRODUCT
2      ቤት          O
3    እቃዎች  I-PRODUCT
4   ማድረቂያ  I-PRODUCT
5   ከማይዝግ          O
6     ብረት          O
7    የተሰራ  B-PRODUCT


In [ ]:
!pip install transformers datasets seqeval accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 949.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 893.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 37.3 MB/s eta 0:00:00


In [74]:
!pip install --upgrade transformers
# Install required packages
!pip install transformers datasets torch seqeval

# **Task 3: Fine-Tuning NER Model**

In [86]:
!pip install -q transformers datasets seqeval torch

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import load_metric, Dataset, DatasetDict
import numpy as np
from sklearn.model_selection import train_test_split

# Configuration
MODEL_NAME = "Davlan/xlm-roberta-base-finetuned-amharic"
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
MAX_LENGTH = 128

# Label mapping (expanded for telegram messages)
label_list = [
    "O",          # Outside
    "B-PRODUCT",  # Beginning of product
    "I-PRODUCT",  # Inside product
    "B-LOC",      # Beginning of location
    "I-LOC",      # Inside location
    "B-PRICE",    # Beginning of price
    "I-PRICE",    # Inside price
    "B-PERSON",   # Beginning of person name
    "I-PERSON"    # Inside person name
]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_conll_data(file_path):
    """Load CoNLL formatted data with error handling."""
    sentences = []
    current_sentence = []
    error_count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    parts = line.split('\t')
                    if len(parts) == 2:
                        token, label = parts
                        current_sentence.append((token, label))
                    else:
                        error_count += 1
                except:
                    error_count += 1
            elif current_sentence:
                sentences.append(current_sentence)
                current_sentence = []

    if error_count > 0:
        print(f"Warning: Skipped {error_count} malformed lines")
    return sentences

def prepare_datasets(file_path):
    """Prepare train/validation/test splits."""
    sentences = load_conll_data(file_path)

    # Convert to token and label lists
    tokens = [[token for token, label in sent] for sent in sentences]
    ner_tags = [[label2id[label] for token, label in sent] for sent in sentences]

    # Split into train (80%), validation (10%), test (10%)
    train_tokens, test_tokens, train_tags, test_tags = train_test_split(
        tokens, ner_tags, test_size=0.2, random_state=42
    )
    train_tokens, val_tokens, train_tags, val_tags = train_test_split(
        train_tokens, train_tags, test_size=0.125, random_state=42  # 0.125*0.8=0.1
    )

    return DatasetDict({
        'train': Dataset.from_dict({'tokens': train_tokens, 'ner_tags': train_tags}),
        'validation': Dataset.from_dict({'tokens': val_tokens, 'ner_tags': val_tags}),
        'test': Dataset.from_dict({'tokens': test_tokens, 'ner_tags': test_tags})
    })

# Load and prepare dataset
datasets = prepare_datasets('/content/merged_conll_file.conll')

def tokenize_and_align_labels(examples):
    """Tokenize and align labels with subword tokens."""
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize all datasets
tokenized_datasets = datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=datasets["train"].column_names
)

# Load model with proper initialization warning handling
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  # This handles the classifier weight mismatch
)

# Training arguments for latest version
training_args = TrainingArguments(
    output_dir="amharic-ner-telegram",
    eval_strategy="epoch",  # Changed from evaluation_strategy to eval_strategy
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

# Metrics calculation
metric = load_metric("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Start training
print("Starting training...")
trainer.train()

# Evaluate on test set
print("\nEvaluating on test set...")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)

# Save the model
trainer.save_model("amharic_ner_telegram_model")
tokenizer.save_pretrained("amharic_ner_telegram_model")

# Save label mappings
import json
with open("amharic_ner_telegram_model/label_mappings.json", "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f)

print("\nModel training complete and saved!")

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at Davlan/xlm-roberta-base-finetuned-amharic and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,1.957573,0.000000,0.000000,0.000000,0.423729
2,No log,1.861694,0.000000,0.000000,0.000000,0.423729
3,No log,1.797537,0.000000,0.000000,0.000000,0.440678
4,No log,1.758958,0.000000,0.000000,0.000000,0.457627
5,No log,1.740778,0.000000,0.000000,0.000000,0.457627



Evaluating on test set...


{'eval_loss': 2.036736488342285, 'eval_precision': 0.046511627906976744, 'eval_recall': 0.044444444444444446, 'eval_f1': 0.045454545454545456, 'eval_accuracy': 0.3026315789473684, 'eval_runtime': 2.8809, 'eval_samples_per_second': 1.736, 'eval_steps_per_second': 0.347, 'epoch': 5.0}

Model training complete and saved!


# **Task 4: Model Comparison & Selection**

In [92]:
!pip install -q transformers datasets seqeval torch pandas

import torch
import pandas as pd
import time
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import load_metric, DatasetDict
import numpy as np
from sklearn.model_selection import train_test_split

# Configuration
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

# Label mapping (must match your dataset)
label_list = ["O", "B-PRODUCT", "I-PRODUCT", "B-LOC", "I-LOC", "B-PRICE", "I-PRICE"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# List of models to compare (with valid HuggingFace model IDs)
MODELS_TO_COMPARE = [
    ("Davlan/xlm-roberta-base-finetuned-amharic", "XLM-Roberta"),
    ("bert-base-multilingual-cased", "mBERT"),
    # Removed bert-tiny-amharic as it's not a valid HF model ID
    # Added alternatives:
    ("distilbert-base-multilingual-cased", "Distil-mBERT"),
    ("xlm-roberta-base", "XLM-R-base")
]

# Initialize training arguments for evaluation
eval_args = TrainingArguments(
    output_dir="./eval_results",
    per_device_eval_batch_size=BATCH_SIZE,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# Metrics calculation
metric = load_metric("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "f1": results["overall_f1"],
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "accuracy": results["overall_accuracy"],
    }

def prepare_test_dataset(file_path):
    """Prepare test dataset from CoNLL file"""
    sentences = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    token, label = line.split('\t')
                    current_sentence.append((token, label))
                except:
                    continue
            elif current_sentence:
                sentences.append(current_sentence)
                current_sentence = []

    # Convert to token and label lists
    tokens = [[token for token, label in sent] for sent in sentences]
    ner_tags = [[label2id[label] for token, label in sent] for sent in sentences]

    return Dataset.from_dict({'tokens': tokens, 'ner_tags': ner_tags})

def evaluate_model(model_name, model_display_name, test_data):
    """Evaluate a single model and return metrics"""
    print(f"\nEvaluating {model_display_name}...")

    try:
        # Load model components
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(label_list),
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True
        )

        # Tokenize and align test data
        def tokenize_and_align(examples):
            tokenized_inputs = tokenizer(
                examples["tokens"],
                truncation=True,
                is_split_into_words=True,
                max_length=MAX_LENGTH,
                padding='max_length'
            )

            labels = []
            for i, label in enumerate(examples["ner_tags"]):
                word_ids = tokenized_inputs.word_ids(batch_index=i)
                previous_word_idx = None
                label_ids = []
                for word_idx in word_ids:
                    if word_idx is None:
                        label_ids.append(-100)
                    elif word_idx != previous_word_idx:
                        label_ids.append(label[word_idx])
                    else:
                        label_ids.append(-100)
                    previous_word_idx = word_idx

                # Pad labels to MAX_LENGTH
                label_ids = label_ids[:MAX_LENGTH] + [-100] * (MAX_LENGTH - len(label_ids))
                labels.append(label_ids)

            tokenized_inputs["labels"] = labels
            return tokenized_inputs

        tokenized_test = test_data.map(tokenize_and_align, batched=True)

        # Create data collator
        data_collator = DataCollatorForTokenClassification(tokenizer)

        # Create trainer
        trainer = Trainer(
            model,
            eval_args,
            eval_dataset=tokenized_test,
            data_collator=data_collator,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics
        )

        # Time inference
        start_time = time.time()
        results = trainer.evaluate()
        inference_time = time.time() - start_time

        # Get model size
        param_size = sum(p.numel() for p in model.parameters())

        return {
            'model': model_display_name,
            'f1': results['eval_f1'],
            'precision': results['eval_precision'],
            'recall': results['eval_recall'],
            'accuracy': results['eval_accuracy'],
            'inference_time': inference_time,
            'model_size': param_size
        }

    except Exception as e:
        print(f"Error evaluating {model_display_name}: {str(e)}")
        return None

def compare_models(test_data_path):
    """Compare multiple models"""
    # Load test data
    test_data = prepare_test_dataset(test_data_path)

    comparison_results = []

    for model_name, model_display_name in MODELS_TO_COMPARE:
        metrics = evaluate_model(model_name, model_display_name, test_data)
        if metrics:
            comparison_results.append(metrics)

    if not comparison_results:
        print("No models were successfully evaluated")
        return None

    # Create comparison dataframe
    df = pd.DataFrame(comparison_results)

    # Normalize metrics for comparison (weights can be adjusted)
    if len(df) > 1:
        # Normalize to 0-1 range
        df['norm_f1'] = (df['f1'] - df['f1'].min()) / (df['f1'].max() - df['f1'].min())
        df['norm_accuracy'] = (df['accuracy'] - df['accuracy'].min()) / (df['accuracy'].max() - df['accuracy'].min())
        df['norm_speed'] = 1 - ((df['inference_time'] - df['inference_time'].min()) /
                               (df['inference_time'].max() - df['inference_time'].min()))
        df['norm_size'] = 1 - ((df['model_size'] - df['model_size'].min()) /
                              (df['model_size'].max() - df['model_size'].min()))

        # Calculate composite score
        df['score'] = (df['norm_f1'] * 0.4 +
                      df['norm_accuracy'] * 0.3 +
                      df['norm_speed'] * 0.2 +
                      df['norm_size'] * 0.1)

        df = df.sort_values('score', ascending=False)

    print("\nModel Comparison Results:")
    print(df[['model', 'f1', 'accuracy', 'inference_time', 'model_size', 'score']].to_markdown())

    if len(df) > 0:
        best_model = df.iloc[0]
        print(f"\nBest model: {best_model['model']} with score {best_model['score']:.3f}")
        print(f"F1: {best_model['f1']:.3f}, Accuracy: {best_model['accuracy']:.3f}")
        print(f"Inference Time: {best_model['inference_time']:.2f}s, Size: {best_model['model_size']:,} parameters")

    return df

# Run comparison with your test data
print("Starting model comparison...")
comparison_results = compare_models('/content/merged_conll_file.conll')

Starting model comparison...

Evaluating XLM-Roberta...


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at Davlan/xlm-roberta-base-finetuned-amharic and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/21 [00:00<?, ? examples/s]


Evaluating mBERT...


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/21 [00:00<?, ? examples/s]


Model Comparison Results:
|    | model        |        f1 |   accuracy |   inference_time |   model_size |    score |
|---:|:-------------|----------:|-----------:|-----------------:|-------------:|---------:|
|  0 | XLM-Roberta  | 0.0560748 |   0.179128 |          8.95607 |    277458439 | 0.666721 |
|  1 | mBERT        | 0         |   0.223647 |          8.55889 |    177268231 | 0.550349 |
|  2 | Distil-mBERT | 0.0289436 |   0.102564 |          8.16779 |    134739463 | 0.506935 |
|  3 | XLM-R-base   | 0.0754226 |   0.076324 |         12.108   |    277458439 | 0.4      |

Best model: XLM-Roberta with score 0.667
F1: 0.056, Accuracy: 0.179
Inference Time: 8.96s, Size: 277,458,439 parameters


# **Task 5: Model Interpretability**

In [97]:
!pip install --upgrade transformers

In [98]:
!pip install shap lime -q

In [128]:
!pip install shap lime -q

import numpy as np
import torch
import shap
from lime import lime_text
from transformers import pipeline

# 1. Load the model with proper configuration
best_model_name = comparison_results.iloc[0]['model']
model_path = "/content/amharic_ner_telegram_model" if best_model_name == "XLM-Roberta" else best_model_name
ner_pipeline = pipeline("ner", model=model_path, tokenizer=model_path, device=0 if torch.cuda.is_available() else -1)

# 2. Define proper label mappings
label_list = ['O', 'B-PRODUCT', 'I-PRODUCT', 'B-LOCATION', 'I-LOCATION', 'B-PRICE', 'I-PRICE']
label2id = {label: i for i, label in enumerate(label_list)}

# 3. Fixed predict_proba for SHAP
def predict_proba(texts):
    """Returns properly shaped probability arrays"""
    batch_probs = []
    max_length = 0  # Track the maximum length of the tokenized input

    for text in texts:
        # Tokenize with the model's tokenizer
        encoding = ner_pipeline.tokenizer(text, return_offsets_mapping=True)
        tokens = ner_pipeline.tokenizer.convert_ids_to_tokens(encoding["input_ids"])
        preds = ner_pipeline(text)

        # Initialize probability matrix
        probs = np.zeros((len(tokens), len(label_list))) + 1e-6  # Small epsilon

        # Align predictions with tokens
        for pred in preds:
            for i, (start, end) in enumerate(encoding["offset_mapping"]):
                if start <= pred['start'] < end:  # Prediction belongs to this token
                    label_idx = label2id.get(pred['entity'], 0)
                    probs[i, label_idx] = pred['score']
                    break

        # Normalize probabilities only if there are non-zero scores
        if probs.sum(axis=1).any():  # Check if any row has a non-zero sum
            probs = probs / probs.sum(axis=1, keepdims=True)

        batch_probs.append(probs)
        max_length = max(max_length, probs.shape[0])  # Update max_length

    # Pad to ensure all outputs have the same shape
    padded_probs = np.zeros((len(batch_probs), max_length, len(label_list))) + 1e-6
    for i, probs in enumerate(batch_probs):
        padded_probs[i, :probs.shape[0], :] = probs

    print(f"Output shapes: {[p.shape for p in batch_probs]}")  # Debugging statement
    return padded_probs

# 4. Sample text for interpretation
sample_text = "ለልጅ አልጋ ዋጋ 2500 ብር በቦሌ አዳራሽ"

# 5. SHAP explanation with proper configuration
explainer = shap.Explainer(
    predict_proba,
    masker=shap.maskers.Text(ner_pipeline.tokenizer),
    output_names=label_list,
    algorithm="permutation"  # Better for NER tasks
)

# Compute SHAP values (for a single sample)
shap_values = explainer([sample_text])

# Visualize for specific entities
print("SHAP values for PRODUCT:")
shap.plots.text(shap_values[..., label2id['B-PRODUCT']])

print("\nSHAP values for LOCATION:")
shap.plots.text(shap_values[..., label2id['B-LOCATION']])

# 6. Improved LIME explanation
explainer_lime = lime_text.LimeTextExplainer(
    class_names=label_list,
    split_expression=lambda x: x.split(),  # Simple word splitting
    bow=False
)

def predict_fn(texts):
    if isinstance(texts, str):
        texts = [texts]

    batch_probs = []
    for text in texts:
        words = text.split()
        preds = ner_pipeline(text)
        probs = np.zeros((len(words), len(label_list)))

        # Align predictions with words
        for pred in preds:
            for i, word in enumerate(words):
                word_start = text.find(word)
                word_end = word_start + len(word)
                if pred['start'] >= word_start and pred['end'] <= word_end:
                    label_idx = label2id.get(pred['entity'], 0)
                    probs[i, label_idx] = pred['score']

        batch_probs.append(probs.flatten())

    return np.array(batch_probs)

# Explain for specific entities
exp = explainer_lime.explain_instance(
    sample_text,
    predict_fn,
    num_features=len(sample_text.split()),
    labels=[label2id['B-PRODUCT'], label2id['B-LOCATION'], label2id['B-PRICE']],
    num_samples=500
)

# Show explanation
exp.show_in_notebook()

Device set to use cpu


Output shapes: [(13, 7), (14, 7), (13, 7), (13, 7), (13, 7)]
Output shapes: [(13, 7), (13, 7), (13, 7), (13, 7), (13, 7)]
Output shapes: [(13, 7), (13, 7), (14, 7), (14, 7), (13, 7)]
Output shapes: [(13, 7), (13, 7), (13, 7), (13, 7), (13, 7)]
Output shapes: [(13, 7), (13, 7), (13, 7)]


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 14 and the array at index 1 has size 13

# **Task 6: FinTech Vendor Scorecard**

In [116]:
import pandas as pd
from datetime import datetime, timedelta

def calculate_vendor_metrics(df):
    """Calculate vendor metrics from scraped data"""
    vendor_metrics = []

    for vendor in df['channel'].unique():
        vendor_data = df[df['channel'] == vendor]

        # Activity & Consistency
        min_date = vendor_data['date'].min()
        max_date = vendor_data['date'].max()
        weeks = (max_date - min_date).days / 7
        posts_per_week = len(vendor_data) / max(1, weeks)

        # Market Reach & Engagement
        avg_views = vendor_data['views'].mean()
        max_views = vendor_data['views'].max()
        top_post = vendor_data[vendor_data['views'] == max_views].iloc[0]

        # Business Profile (from NER)
        # Note: In practice you'd run NER on all posts
        # Here we'll simulate with some dummy data
        avg_price = 1000  # Would be calculated from NER results
        product_variety = 20  # Would count unique products from NER

        # Create score (weights can be adjusted)
        lending_score = (avg_views * 0.4 +
                        posts_per_week * 0.3 +
                        avg_price * 0.2 +
                        product_variety * 0.1)

        vendor_metrics.append({
            'vendor': vendor,
            'posting_frequency': posts_per_week,
            'avg_views': avg_views,
            'max_views': max_views,
            'top_post_text': top_post['processed_text'],
            'top_post_date': top_post['date'],
            'avg_price': avg_price,
            'product_variety': product_variety,
            'lending_score': lending_score
        })

    return pd.DataFrame(vendor_metrics)

def generate_scorecard(vendor_metrics):
    """Generate a vendor scorecard"""
    scorecard = vendor_metrics.sort_values('lending_score', ascending=False)

    print("Vendor Lending Scorecard")
    print("="*50)
    for idx, row in scorecard.iterrows():
        print(f"\nVendor: {row['vendor']}")
        print(f"Lending Score: {row['lending_score']:.1f}")
        print(f"Posting Frequency: {row['posting_frequency']:.1f} posts/week")
        print(f"Average Views: {row['avg_views']:.0f}")
        print(f"Top Post Views: {row['max_views']} on {row['top_post_date']}")
        print(f"Average Price: {row['avg_price']} ETB")
        print(f"Product Variety: {row['product_variety']} unique products")

    return scorecard

# Load scraped data
df = pd.read_csv('/content/telegram_data_processed_task1.csv')
df['date'] = pd.to_datetime(df['date'])

# Calculate metrics
vendor_metrics = calculate_vendor_metrics(df)

# Generate scorecard
scorecard = generate_scorecard(vendor_metrics)

# Save results
scorecard.to_csv('vendor_scorecard.csv', index=False)

Vendor Lending Scorecard

Vendor: Leyueqa
Lending Score: 17174.9
Posting Frequency: 9.5 posts/week
Average Views: 42425
Top Post Views: 201645 on 2024-01-26 07:01:53+00:00
Average Price: 1000 ETB
Product Variety: 20 unique products

Vendor: MerttEka
Lending Score: 10382.4
Posting Frequency: 14.4 posts/week
Average Views: 25440
Top Post Views: 216507 on 2024-06-23 05:33:57+00:00
Average Price: 1000 ETB
Product Variety: 20 unique products

Vendor: belaclassic
Lending Score: 5590.0
Posting Frequency: 6.6 posts/week
Average Views: 13465
Top Post Views: 24312 on 2024-11-26 04:59:16+00:00
Average Price: 1000 ETB
Product Variety: 20 unique products

Vendor: ZemenExpress
Lending Score: 5066.2
Posting Frequency: 17.3 posts/week
Average Views: 12148
Top Post Views: 29997 on 2022-09-15 12:39:28+00:00
Average Price: 1000 ETB
Product Variety: 20 unique products

Vendor: AwasMart
Lending Score: 3798.2
Posting Frequency: 22.6 posts/week
Average Views: 8974
Top Post Views: 30238 on 2022-09-24 11:33:10